# TabICL: A Tabular Foundation Model for In-Context Learning on Large Data (ICML 2025)

[Link to documentation](https://github.com/soda-inria/tabicl)

This notebook is a slightly modified fork of Alana Monks' copy of my previous baseline notebook using TabICL model. Thanks to Alana for the excellent feature engineering!

> ⚠️ **Sebelum mulai:** Pastikan sudah upload `train.csv` dan `test.csv` secara manual ke Google Colab (klik ikon folder di sidebar kiri, lalu drag & drop file).

# Imports and Configuration

In [ ]:
!pip install --quiet tabicl

In [ ]:
import warnings
import numpy as np
import pandas as pd
from tabicl import TabICLClassifier

warnings.filterwarnings('ignore')

RANDOM_STATE = 777

# Data Loading

In [ ]:
# #i just made this to display all cols so i could decide on features to engineer.
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", 250)

# for c in train.columns:
#     print("\n" + "=" * 80)
#     print(c)
#     print(train[c].head(3).to_string(index=True))

**Alana's part :)**

**This next part is my contribution where I engineer the following features into the datasets😊:**

**Pledge Related** (I figured this was an important to capture interactions it has with other features)
- pledge_total = pledge_limited + pledge_unlimited
- pledge_limited_share = pledge_limited / (pledge_total + eps)
- pledge_unlimited_share = pledge_unlimited / (pledge_total + eps)
- pledge_control_gap = share_pledge_control - pledge_total
- pledge_control_ratio = share_pledge_control / (pledge_total + eps)
- pledge_any = 1{pledge_total > 0}
- control_pledge_any = 1{share_pledge_control > 0}


**Market stress interactions**
- abs_ret_1y = abs(ret_1y)
- downside_ret_1y = min(ret_1y, 0)
- vol_x_absret = volatility * abs_ret_1y
- turnover_x_absret = annual_turnover * abs_ret_1y
- turnover_x_vol = annual_turnover * volatility


In [ ]:
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

train = train.replace(",", "", regex=True)
test = test.replace(",", "", regex=True)

eps = 1e-6

for df in (train, test):
    ret_1y = pd.to_numeric(df["Stock price rise and fall in the last year"], errors="coerce").fillna(0.0)
    volatility = pd.to_numeric(df["Stock Volatility"], errors="coerce").fillna(0.0)
    annual_turnover = pd.to_numeric(df["Annual turnover rate"], errors="coerce").fillna(0.0)
    pe = pd.to_numeric(df["P/E ratio"], errors="coerce").fillna(0.0)
    pb = pd.to_numeric(df["P/B ratio"], errors="coerce").fillna(0.0)

    abs_ret_1y = ret_1y.abs()
    downside_ret_1y = np.minimum(ret_1y, 0.0)

    df["abs_ret_1y"] = abs_ret_1y
    df["downside_ret_1y"] = downside_ret_1y
    df["vol_x_absret"] = volatility * abs_ret_1y
    df["turnover_x_absret"] = annual_turnover * abs_ret_1y
    df["turnover_x_vol"] = annual_turnover * volatility

    pe_slog = np.sign(pe) * np.log1p(np.abs(pe))
    pb_slog = np.sign(pb) * np.log1p(np.abs(pb))

    df["pe_slog"] = pe_slog
    df["pb_slog"] = pb_slog
    df["val_mix"] = pe_slog * pb_slog
    df["val_ratio"] = pe_slog / (np.abs(pb_slog) + eps)

    roa = pd.to_numeric(df["ROA"], errors="coerce").fillna(0.0)
    roe = pd.to_numeric(df["ROE"], errors="coerce").fillna(0.0)

    cash_income = pd.to_numeric(df["Cash income ratio"], errors="coerce").fillna(0.0)
    cash_income_3y = pd.to_numeric(df["Average cash income ratio in recent three years"], errors="coerce").fillna(0.0)


X = train.drop(columns=["Stock code", "IsDefault"]).astype(float)
y = train["IsDefault"].astype(int)
X_test = test.drop(columns=["Stock code"]).astype(float)

In [ ]:
X.shape

In [ ]:
!pip install --quiet imbalanced-learn
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=RANDOM_STATE)
X, y = smote.fit_resample(X, y)
X.shape

In [ ]:
!pip install --quiet ctgan
from ctgan import CTGAN

# Выделяем только миноритарные примеры
X_min = X[y == 1]

# Обучаем генератор
model_ctgan = CTGAN(epochs=10)  # настройте под свои данные
model_ctgan.fit(pd.DataFrame(X_min))

# Генерируем нужное количество новых примеров
n_new = 500
synthetic = model_ctgan.sample(n_new)
synthetic['IsDefault'] = 1

# Объединяем с исходными данными (можно и с мажоритарными)
X = pd.concat([X, synthetic.drop('IsDefault', axis=1)], ignore_index=True)
y = pd.concat([y, synthetic['IsDefault']], ignore_index=True)


X_min = X[y == 0]

# Обучаем генератор
model_ctgan = CTGAN(epochs=10)  # настройте под свои данные
model_ctgan.fit(pd.DataFrame(X_min))

# Генерируем нужное количество новых примеров
n_new = 500
synthetic = model_ctgan.sample(n_new)
synthetic['IsDefault'] = 0

# Объединяем с исходными данными (можно и с мажоритарными)
X = pd.concat([X, synthetic.drop('IsDefault', axis=1)], ignore_index=True)
y = pd.concat([y, synthetic['IsDefault']], ignore_index=True)

X.shape

# Training

In [ ]:
%%time

# Cek parameter yang didukung versi tabicl yang terinstall
import inspect
valid_params = inspect.signature(TabICLClassifier.__init__).parameters

kwargs = dict(
    n_estimators=256,
    norm_methods=["quantile"],
    feat_shuffle_method="latin",
    class_shuffle_method="shift",
    outlier_threshold=4.0,
    softmax_temperature=0.8,
    average_logits=False,
    support_many_classes=True,
    model_path=None,
    allow_auto_download=True,
    checkpoint_version="tabicl-classifier-v1.1-20250506.ckpt",
    device=None,
    use_amp="auto",
    use_fa3="auto",
    offload_mode="auto",
    disk_offload_dir=None,
    random_state=RANDOM_STATE,
    n_jobs=None,
    verbose=False,
    inference_config=None,
)

# Tambahkan parameter opsional hanya jika didukung versi ini
optional_params = {
    'size': 8,
}
for k, v in optional_params.items():
    if k in valid_params:
        kwargs[k] = v

# Filter hanya param yang ada di versi ini
kwargs = {k: v for k, v in kwargs.items() if k in valid_params}

model = TabICLClassifier(**kwargs)
model.fit(X, y)

In [ ]:
%%time
test_preds = model.predict_proba(X_test)

# Submission

In [ ]:
sub = pd.DataFrame({
    'Stock code': test['Stock code'],
    'IsDefault': test_preds[:, 1]
})
sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
# Download submission file ke komputer lokal
from google.colab import files
files.download('submission.csv')